# Split Generation

## Scientific objective
Persist random diagnostic splits, a primary global scaffold split, repeated scaffold splits, and hooks for temporal/external validation without molecule or scaffold leakage.

## Inputs
- `data/processed/endpoint_records.csv`

## Expected outputs
- `data/processed/split_assignments.csv`
- `data/processed/modeling_records.csv`
- `data/processed/repeated_scaffold_splits.csv`
- split audit

## Dependencies
pandas, NumPy

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
All endpoints share one molecule-level assignment. Missing endpoint labels do not alter the global partition.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
A scaffold split is still not a prospective or independent external validation. Empty/acyclic scaffold grouping should be inspected for imbalance.

## Next notebook
[09_feature_generation.ipynb](./09_feature_generation.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from toxicity_screening.pipeline import generate_global_splits
from toxicity_screening.splitting import (
    repeated_scaffold_assignments,
    assert_no_group_leakage,
)

assignments = generate_global_splits(ROOT)

records = pd.read_parquet(
    ROOT
    / "data"
    / "processed"
    / "modeling_records.parquet"
)

assert_no_group_leakage(
    records,
    "molecule_id",
    "scaffold_split",
)

assert_no_group_leakage(
    records,
    "scaffold",
    "scaffold_split",
)

print(
    "Split assignments:",
    assignments.shape,
)

print(
    records["scaffold_split"].value_counts()
)

print(
    "No molecule or scaffold leakage detected."
)

Split assignments: (25583, 6)
scaffold_split
train         36108
test           7586
validation     6918
Name: count, dtype: Int64
No molecule or scaffold leakage detected.


In [3]:
seeds = CONFIGS["data_config"]["splits"]["repeated_scaffold_seeds"]
repeated = repeated_scaffold_assignments(assignments, seeds, scaffold_column="scaffold")
repeated.to_csv(ROOT / "data/processed/repeated_scaffold_splits.csv", index=False)
audit = records.groupby(["endpoint","scaffold_split"]).agg(records=("molecule_id","size"), observed=("label","count"), positives=("label",lambda x:int((x==1).sum()))).reset_index()
audit.to_csv(ROOT / "data/metadata/split_audit.csv", index=False)
assert set(records.scaffold_split)=={"train","validation","test"}
display(audit)

,endpoint,scaffold_split,records,observed,positives
0,SR-ARE,test,1142,847,174
1,SR-ARE,train,5527,4139,620
2,SR-ARE,validation,927,680,112
3,SR-ATAD5,test,1147,1027,66
4,SR-ATAD5,train,5534,5013,158
5,SR-ATAD5,validation,929,829,36
6,SR-MMP,test,1147,842,186
7,SR-MMP,train,5529,4130,596
8,SR-MMP,validation,928,677,114
9,SR-p53,test,1147,986,84


### Completion gate
Confirm that the declared artifacts exist before continuing to `09_feature_generation.ipynb`.